# Maximum Causal Entropy IRL on Bus Engine Data

This notebook demonstrates:
1. Loading the Rust bus engine replacement data
2. Fitting MCE IRL to recover reward parameters
3. Computing standard errors and confidence intervals
4. Making predictions using the learned model

In [1]:
import numpy as np
import pandas as pd
from econirl.datasets import load_rust_bus
from econirl import MCEIRL

## 1. Load the Data

In [2]:
df = load_rust_bus()
print(f"Observations: {len(df):,}")
print(f"Buses: {df['bus_id'].nunique()}")
print(f"Replacement rate: {df['replaced'].mean():.2%}")
df.head(10)

Observations: 9,410
Buses: 90
Replacement rate: 5.13%


,bus_id,period,mileage,mileage_bin,replaced,group
0,1,1,0.0,0,0,1
1,1,2,0.0,0,0,1
2,1,3,5.0,1,0,1
3,1,4,5.0,1,0,1
4,1,5,5.0,1,0,1
5,1,6,5.0,1,0,1
6,1,7,10.0,2,0,1
7,1,8,15.0,3,0,1
8,1,9,15.0,3,0,1
9,1,10,15.0,3,0,1


## 2. Analyze Replacement Patterns

In [3]:
# Replacement rate by mileage bin
print("Replacement rate by mileage bin:")
print("-" * 40)
for b in range(0, 90, 10):
    subset = df[(df['mileage_bin'] >= b) & (df['mileage_bin'] < b + 10)]
    if len(subset) > 0:
        print(f"Bins {b:2d}-{b+9}: {subset['replaced'].mean():6.2%} ({len(subset):4d} obs)")

Replacement rate by mileage bin:
----------------------------------------
Bins  0-9:  4.77% (6149 obs)
Bins 10-19:  5.49% (2260 obs)
Bins 20-29:  5.82% ( 704 obs)
Bins 30-39:  8.30% ( 229 obs)
Bins 40-49:  6.56% (  61 obs)
Bins 50-59: 28.57% (   7 obs)


## 3. Fit MCE IRL Model

In [4]:
# Define state features (normalized mileage)
n_states = 90
features = np.arange(n_states).reshape(-1, 1) / 100

# Create and fit the model
model = MCEIRL(
    n_states=n_states,
    n_actions=2,
    discount=0.99,
    feature_matrix=features,
    feature_names=["mileage_cost"],
    se_method="hessian",
    inner_max_iter=5000,
    verbose=True,
)

model.fit(
    data=df,
    state="mileage_bin",
    action="replaced",
    id="bus_id",
)

[MCE IRL (Ziebart 2010)] Empirical features: tensor([0.0862])
[MCE IRL (Ziebart 2010)] Initial distribution entropy: -0.000
[MCE IRL (Ziebart 2010)] Starting MCE IRL with gradient ascent (lr=0.1)


[MCE IRL (Ziebart 2010)] Iter 10: obj=0.002726, ||grad||=0.073832


[MCE IRL (Ziebart 2010)] Iter 20: obj=0.002724, ||grad||=0.073812


[MCE IRL (Ziebart 2010)] Iter 30: obj=0.002723, ||grad||=0.073791


[MCE IRL (Ziebart 2010)] Iter 40: obj=0.002721, ||grad||=0.073770


...

[MCE IRL (Ziebart 2010)] Iter 180: obj=0.001951, ||grad||=0.062469


[MCE IRL (Ziebart 2010)] Iter 190: obj=0.000128, ||grad||=0.015988


[MCE IRL (Ziebart 2010)] Iter 200: obj=0.000001, ||grad||=0.001388
[MCE IRL (Ziebart 2010)] Computing standard errors via hessian


[MCE IRL (Ziebart 2010)] Optimization complete: feature_diff=0.001388, LL=-5508.19


MCEIRL(n_states=90, n_actions=2, discount=0.99, fitted=True)

## 4. Results and Inference

In [5]:
print(model.summary())

                  Maximum Causal Entropy IRL Results                  
Method:                   MCE IRL (Ziebart 2010)
Discount Factor (β):      0.99
No. States:               90
No. Actions:              2
Log-Likelihood:           -5,508.19
Converged:                No
----------------------------------------------------------------------

Parameter Estimates:
----------------------------------------------------------------------
Parameter                Estimate      Std Err     t-stat               95% CI
----------------------------------------------------------------------
mileage_cost               1.3643       0.0051     270.02     [1.3544, 1.3742]
----------------------------------------------------------------------


In [6]:
# Parameter estimates with confidence intervals
print("Parameter Estimates:")
print("-" * 50)
for name, val in model.params_.items():
    se = model.se_.get(name, np.nan)
    ci_low = val - 1.96 * se
    ci_high = val + 1.96 * se
    print(f"{name}: {val:.4f} (SE: {se:.4f})")
    print(f"  95% CI: [{ci_low:.4f}, {ci_high:.4f}]")

Parameter Estimates:
--------------------------------------------------
mileage_cost: 1.3643 (SE: 0.0051)
  95% CI: [1.3544, 1.3742]


## 5. Predictions

In [7]:
# Predict choice probabilities at different states
test_states = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 89])
proba = model.predict_proba(test_states)

print(f"{'State':>6} {'P(keep)':>10} {'P(replace)':>12}")
print("-" * 30)
for i, s in enumerate(test_states):
    print(f"{s:>6} {proba[i, 0]:>10.4f} {proba[i, 1]:>12.4f}")

 State    P(keep)   P(replace)
------------------------------
     0     0.5000       0.5000
    10     0.9786       0.0214
    20     1.0000       0.0000
    30     1.0000       0.0000
    40     1.0000       0.0000
    50     1.0000       0.0000
    60     1.0000       0.0000
    70     1.0000       0.0000
    80     1.0000       0.0000
    89     1.0000       0.0000


## 6. Compare to Empirical Data

In [8]:
# Compare predicted vs empirical replacement probabilities
print(f"{'State':>6} {'Empirical':>12} {'Predicted':>12} {'Diff':>10}")
print("-" * 45)
for i, s in enumerate(test_states):
    subset = df[df['mileage_bin'] == s]
    pred_p = proba[i, 1]
    if len(subset) > 0:
        emp_p = subset['replaced'].mean()
        diff = pred_p - emp_p
        print(f"{s:>6} {emp_p:>12.4f} {pred_p:>12.4f} {diff:>+10.4f}")
    else:
        print(f"{s:>6} {'N/A':>12} {pred_p:>12.4f}")

 State    Empirical    Predicted       Diff
---------------------------------------------
     0       0.0411       0.5000    +0.4589
    10       0.0546       0.0214    -0.0332
    20       0.0800       0.0000    -0.0800
    30       0.1579       0.0000    -0.1579
    40       0.0909       0.0000    -0.0909
    50       0.0000       0.0000    +0.0000
    60          N/A       0.0000
    70          N/A       0.0000
    80          N/A       0.0000
    89          N/A       0.0000


## 7. Recovered Reward Function

In [9]:
# Show the recovered reward at different states
if model.reward_ is not None:
    r0 = model.reward_[0]
    print(f"{'State':>6} {'Mileage':>10} {'R(s)':>12} {'R(s) - R(0)':>14}")
    print("-" * 45)
    for s in test_states:
        r = model.reward_[s]
        print(f"{s:>6} {s*5:>10}k {r:>12.4f} {r - r0:>14.4f}")

 State    Mileage         R(s)    R(s) - R(0)
---------------------------------------------
     0          0k       0.0000         0.0000
    10         50k       0.1364         0.1364
    20        100k       0.2729         0.2729
    30        150k       0.4093         0.4093
    40        200k       0.5457         0.5457
    50        250k       0.6821         0.6821
    60        300k       0.8186         0.8186
    70        350k       0.9550         0.9550
    80        400k       1.0914         1.0914
    89        445k       1.2142         1.2142


In [10]:
print(f"\nModel converged: {model.converged_}")
print(f"Log-likelihood: {model.log_likelihood_:.2f}")


Model converged: False
Log-likelihood: -5508.19
